In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Arc

import paulssonlab.deaton.trenchripper.trenchripper as tr
import numpy as np
import pandas as pd
import seaborn as sns
import scipy as sp
import sklearn as skl
import dask.dataframe as dd
import dask.array as da
import colorcet as cc
import dask
import warnings
import copy
import random
from sklearn.metrics.pairwise import euclidean_distances,manhattan_distances,cosine_distances

from sklearn.metrics import silhouette_score
import scipy.stats
from sklearn.linear_model import LinearRegression
from sklearn.cluster import AffinityPropagation
from sklearn.manifold import TSNE
from sklearn.cluster import AgglomerativeClustering

import scanpy as sc
import anndata
import scipy as sp
import scipy.sparse
import dask.array as da
#from igraph.drawing.text import TextDrawer
from tslearn.neighbors import KNeighborsTimeSeries
from tslearn.metrics import cdist_soft_dtw_normalized,cdist_soft_dtw
import networkx as nx
import igraph as ig
import leidenalg
import umap
from scanpy.plotting.palettes import default_20,vega_20_scanpy
from matplotlib import pyplot as plt
from matplotlib import cm
import ast
from tslearn.barycenters import softdtw_barycenter,dtw_barycenter_averaging,euclidean_barycenter
import xarray as xr

import pylab
import scipy.cluster.hierarchy as sch

import matplotlib.gridspec as gridspec
import matplotlib as mpl

import re
import goatools
import goatools.base
from goatools.base import download_go_basic_obo

from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
from goatools.semantic import semantic_similarity
from goatools.semantic import TermCounts, get_info_content

from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS
from goatools.go_enrichment import GOEnrichmentStudy

import holoviews as hv
hv.extension('bokeh')
random.seed(42)
np.random.seed(42)

warnings.filterwarnings(action='once',category=UserWarning)

In [ ]:
short_expt_strong_effect_threshold = 1.5
long_expt_strong_effect_threshold = 1.5

##override with short for now
short_expt_rootpath = "/home/de64/group/CRISPRi_Libraries/dev_notebooks/2023-02-26_lDE20_10Hour_Analysis"
long_expt_rootpath = "/home/de64/group/CRISPRi_Libraries/dev_notebooks/2023-01-02_lDE23_32Hour_Analysis"
short_expt_z_score_thr_dir = short_expt_rootpath + "/Z_Score_Thr_" + str(short_expt_strong_effect_threshold) + "_N_Neighbors_10"
long_expt_z_score_thr_dir = long_expt_rootpath + "/Z_Score_Thr_" + str(long_expt_strong_effect_threshold) + "_N_Neighbors_10"

short_expt_clustree_pandas_df = pd.read_pickle(short_expt_z_score_thr_dir+"/Pandas_Dataframe.pkl")
short_expt_an_df_clustree_r_compat = anndata.read_h5ad(short_expt_z_score_thr_dir+"/AnnData.h5ad")
long_expt_clustree_pandas_df = pd.read_pickle(long_expt_z_score_thr_dir+"/Pandas_Dataframe.pkl")
long_expt_an_df_clustree_r_compat = anndata.read_h5ad(long_expt_z_score_thr_dir+"/AnnData.h5ad")

ss_mean_cv_df = pd.read_pickle("/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2023-01-18_lDE20_Lineage_Cell_Cycle_Timeseries_Steady_State_MeanCV.pkl")
ss_mean_cv_df['Mean: Delta time (hr)'] = ss_mean_cv_df['Mean: Delta time (s)']/3600
ss_mean_cv_df['Variance: Delta time (hr)'] = ss_mean_cv_df['Variance: Delta time (s)']/(3600**2)
control_ss_mean_cv_df = ss_mean_cv_df[ss_mean_cv_df["Category"].isin(['OnlyPlasmid', 'NoTarget'])]

In [ ]:
## remember this has already been filtered for clustering

long_expt_length_width_df = long_expt_clustree_pandas_df[['Kernel Trace: Delta: Volume','Kernel Trace: Division: major_axis_length',\
                                                          'Kernel Trace: minor_axis_length',"Kernel Trace: Instantaneous Growth Rate: Volume"]]
short_expt_length_width_df = short_expt_clustree_pandas_df[['Kernel Trace: Delta: Volume','Kernel Trace: Division: major_axis_length',\
                                                          'Kernel Trace: minor_axis_length',"Kernel Trace: Instantaneous Growth Rate: Volume"]]

long_expt_growth_df_t0 = long_expt_length_width_df.apply(lambda x: [column[0] for _,column in x.iteritems()])
short_expt_growth_df_t0 = short_expt_length_width_df.apply(lambda x: [column[0] for _,column in x.iteritems()])

long_expt_growth_df_t_24hr_post = long_expt_length_width_df.apply(lambda x: [column[31] for _,column in x.iteritems()])
short_expt_growth_df_t_8hr_post = short_expt_length_width_df.apply(lambda x: [column[19] for _,column in x.iteritems()])

short_expt_growth_df_t_8hr_post["Gene"] = short_expt_clustree_pandas_df["Gene"]
short_expt_growth_df_t_8hr_post["TargetID"] = short_expt_clustree_pandas_df["TargetID"]

### Preprocessing Time Slices 

In [ ]:
kernel_label_list = ['Kernel Trace: Delta time (hr)','Kernel Trace: Septum Displacement Length Normalized','Kernel Trace: major_axis_length',\
'Kernel Trace: minor_axis_length','Kernel Trace: mCherry mean_intensity','Kernel Trace: Instantaneous Growth Rate: Volume']
zscore_kernel_label_list = [item + ": Transformed: z score" for item in kernel_label_list]

time_slices = [11,15,19]
time_slice_labels = ["4 Hours", "6 Hours", "8 Hours"]
short_expt_clustree_pandas_df['Kernel Trace: Delta time (hr): Transformed: z score'] = short_expt_clustree_pandas_df['Kernel Trace: Delta time (s): Transformed: z score']

for time_slice_idx,time_slice in enumerate(time_slices):
    for kernel_label in kernel_label_list:
        zscore_kernel_label = kernel_label + ": Transformed: z score"
        
        short_expt_clustree_pandas_df[time_slice_labels[time_slice_idx] + ": " + kernel_label] = \
            short_expt_clustree_pandas_df[kernel_label].apply(lambda x: x[time_slice])
        
        short_expt_clustree_pandas_df[time_slice_labels[time_slice_idx] + ": " + zscore_kernel_label] = \
            short_expt_clustree_pandas_df[zscore_kernel_label].apply(lambda x: x[time_slice])

### Figure A
- Use leiden 0.5 as the low resolution view
- Note which groups will be hilighted in illustrator
- Highlighted groups will be associated with GO

In [ ]:
clusters_df = short_expt_clustree_pandas_df.dropna(subset=["L0"])

In [ ]:
cbar_label = ""
labels = ['Mean: Delta time (hr)','Mean: Septum Displacement Length Normalized','Mean: major_axis_length',\
          'Mean: minor_axis_length','Mean: mCherry mean_intensity','Mean: Instantaneous Growth Rate: Volume']

kernel_label_list = ['Kernel Trace: Delta time (hr)','Kernel Trace: Septum Displacement Length Normalized','Kernel Trace: major_axis_length',\
'Kernel Trace: minor_axis_length','Kernel Trace: mCherry mean_intensity','Kernel Trace: Instantaneous Growth Rate: Volume']

title_list = ["Interdivision Time (hr)","Septum Displacement/Length","Length $(\mu m)$","Width $(\mu m)$",\
              "mKate2 Mean Intensity (AU)","Growth Rate (1/hr)"]

all_labels = labels

median_vals_dict = control_ss_mean_cv_df[labels].median().to_dict()
median_properties = clusters_df.groupby("L1")[labels].median()

min_vals = []
max_vals = []
norm_list = []
for label in all_labels:
    vmin,vmax = np.nanpercentile(median_properties[label],10),np.nanpercentile(median_properties[label],90)
    vcenter = median_vals_dict[label]
    min_deviation,max_deviation = (vcenter-vmin,vmax-vcenter)
    if min_deviation>max_deviation:
        vmax = vcenter + min_deviation
    else:
        vmin = vcenter - max_deviation
    min_vals.append(vmin)
    max_vals.append(vmax)

In [ ]:
fig, axes = plt.subplots(1, len(labels)+1, figsize=(4, 6), sharey=True)

label_names = [r'$ \tau_{C} $',r'$ L_{Sept} $',r'$ L_{C} $',r'$ W_{C} $',r'$ I_{rpsL} $',r'$ \lambda $',]
# Create separate single-column heatmaps for each column
# sns.heatmap(np.array(median_properties.index.tolist()).reshape(-1,1).astype(int),cmap=vega_20_scanpy,\
#             cbar=False, yticklabels=False, xticklabels=False, ax=axes[0],annot=True,\
#                linewidths=0,linecolor="black", annot_kws={"fontsize":14})

white_cmap = mpl.colors.ListedColormap(['white'])

# Create separate single-column heatmaps for each column
sns.heatmap(np.array(median_properties.index.tolist()).reshape(-1,1).astype(int),cmap=white_cmap,\
            cbar=False, yticklabels=False, xticklabels=False, ax=axes[0],annot=True,\
               linewidths=0,linecolor="black", annot_kws={"fontsize":14},alpha=0)

cbar_list = []
for i, label in enumerate(labels):
    working_heatmap = sns.heatmap(median_properties.values[:,i:i+1],vmin=min_vals[i],vmax=max_vals[i],\
                cmap="coolwarm",center=median_vals_dict[label],\
               cbar=False, yticklabels=False, ax=axes[i+1],annot=False,\
               linewidths=0.5,linecolor="black")
    axes[i+1].set_xticklabels(labels=[label_names[i]],fontsize=16)
# Remove margins between subplots
plt.subplots_adjust(wspace=0)
# plt.savefig("/home/de64/group/CRISPRi_Libraries/dev_notebooks/2023-04-05_Multi_Dataset_Figures/Global_Clusters.png",dpi=300,transparent=True)

In [ ]:
fig, axes = plt.subplots(3, len(labels)//3, figsize=(2, 6))
for i,label in enumerate(labels):
    # Create a colormap and a normalization object
    cmap = plt.get_cmap('coolwarm')  # or any other colormap, e.g., 'jet', 'coolwarm', 'plasma', etc.
    norm = mpl.colors.Normalize(vmin=min_vals[i], vmax=max_vals[i])  # set the range of values to normalize, e.g., 0 to 1

    # Create a ScalarMappable object
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])  # this line is needed to create the ScalarMappable object, but the array can be empty
    cbar = plt.colorbar(sm, cax=axes[i//2,i%2], orientation='vertical')
    axes[i//2,i%2].set_title(label_names[i])

# Show the colorbar
plt.tight_layout()
# plt.savefig("/home/de64/group/CRISPRi_Libraries/dev_notebooks/2023-04-05_Multi_Dataset_Figures/Global_Clusters_Legend.png",dpi=300,transparent=True)

### Filamentous phenotypes Luis

In [ ]:
min_representatives = 10
cluster_level = "L1"
subcluster_level = "L2"
selected_clusters = ["4","5","18"]
highlighted_subclusters = ["12","17","25","30"] #3 for ribo

long_cluster_df = clusters_df[clusters_df[cluster_level].isin(selected_clusters)].copy()
clusters,n_rep_clusters = np.unique(long_cluster_df[subcluster_level],return_counts=True)
subclusters = clusters[n_rep_clusters>=min_representatives]
long_cluster_df = clusters_df[clusters_df[subcluster_level].isin(subclusters)].copy()
long_cluster_df.loc[:,subcluster_level] = long_cluster_df[subcluster_level].cat.remove_unused_categories().copy()

long_median_properties_list = []
for time_slice_idx,time_slice in enumerate(time_slices):
    sliced_kernel_label_list = [time_slice_labels[time_slice_idx] + ": " + kernel_label for kernel_label in kernel_label_list]
    long_median_properties = long_cluster_df.groupby(subcluster_level)[sliced_kernel_label_list].median()
    long_median_properties_list.append(long_median_properties)

In [ ]:
unhighlighted_subclusters = [item for item in long_median_properties.index.tolist() if item not in highlighted_subclusters]
# Generate a categorical colormap using colorcet
colors = cc.glasbey_light[:len(long_median_properties.index)]
# Create a dictionary mapping each category to a color
color_dict = dict(zip(long_median_properties.index, colors))
reorder_cluster_dict = {cluster:i for i,cluster in enumerate(long_median_properties.index.tolist())}
for subcluster in unhighlighted_subclusters:
    color_dict[subcluster] = '#808080'
categorical_colors = [color_dict[item] for item in long_median_properties.index.tolist()]
cmap = sns.color_palette(categorical_colors)
# normalize the data
norm = mpl.colors.Normalize(vmin=0, vmax=len(reorder_cluster_dict)-1)

In [ ]:
# I can generate the colorbars

In [ ]:
label_names

In [ ]:
fig, axes = plt.subplots(1, len(labels)+1, figsize=(4, 1), sharey=True)
# fig.suptitle(time_slice_labels[time_slice_idx], fontsize=16)
plt.subplots_adjust(wspace=0)

# Create separate single-column heatmaps for each column
sns.heatmap(np.array([3]).reshape(-1,1),cmap=cmap,norm=norm,\
            cbar=False, yticklabels=False, xticklabels=False, ax=axes[0],annot=False,\
               linewidths=0,linecolor="black", annot_kws={"fontsize":14})

cbar_list = []
for i, label in enumerate(labels):
    working_heatmap = sns.heatmap(long_median_properties.values[1:2,i:i+1],vmin=min_vals[i],vmax=max_vals[i],\
                cmap="coolwarm",center=median_vals_dict[label],\
               cbar=False, yticklabels=False, ax=axes[i+1],annot=False,\
               linewidths=0.5,linecolor="black")
    axes[i+1].set_xticklabels(labels=[label_names[i]],fontsize=16)
plt.subplots_adjust(wspace=0)

In [ ]:
long_median_properties.index

In [ ]:
labels

In [ ]:
fig, axes = plt.subplots(1, len(labels)+1, figsize=(4, 4), sharey=True)
fig.suptitle(time_slice_labels[time_slice_idx], fontsize=16)

label_names = [r'$ \tau_{C} $',r'$ L_{Sept} $',r'$ L_{C} $',r'$ W_{C} $',r'$ I_{rpsL} $',r'$ \lambda $',]

long_median_properties = long_median_properties_list[time_slice_idx]

cluster_colors = {int(long_median_properties.index.tolist()[i]):vega_20_scanpy[i] for i in range(len(long_median_properties))}
# white_cmap = mpl.colors.ListedColormap(['white'])

# Create separate single-column heatmaps for each column
sns.heatmap(np.array(range(len(long_median_properties))).reshape(-1,1),cmap=cmap,norm=norm,\
            cbar=False, yticklabels=False, xticklabels=False, ax=axes[0],annot=False,\
               linewidths=0,linecolor="black", annot_kws={"fontsize":14})

cbar_list = []
for i, label in enumerate(labels):
    working_heatmap = sns.heatmap(long_median_properties.values[:,i:i+1],vmin=min_vals[i],vmax=max_vals[i],\
                cmap="coolwarm",center=median_vals_dict[label],\
               cbar=False, yticklabels=False, ax=axes[i+1],annot=False,\
               linewidths=0.5,linecolor="black")
    axes[i+1].set_xticklabels(labels=[label_names[i]],fontsize=16)

# Remove margins between subplots
plt.subplots_adjust(wspace=0)

### End of: Luis

### Figure B: Filamentous Phenotypes

In [ ]:
min_representatives = 10
cluster_level = "L1"
subcluster_level = "L2"
selected_clusters = ["4","5","18"]
highlighted_subclusters = ["12","17","25","30"] #3 for ribo

long_cluster_df = clusters_df[clusters_df[cluster_level].isin(selected_clusters)].copy()
clusters,n_rep_clusters = np.unique(long_cluster_df[subcluster_level],return_counts=True)
subclusters = clusters[n_rep_clusters>=min_representatives]
long_cluster_df = clusters_df[clusters_df[subcluster_level].isin(subclusters)].copy()
long_cluster_df.loc[:,subcluster_level] = long_cluster_df[subcluster_level].cat.remove_unused_categories().copy()

long_median_properties_list = []
for time_slice_idx,time_slice in enumerate(time_slices):
    sliced_kernel_label_list = [time_slice_labels[time_slice_idx] + ": " + kernel_label for kernel_label in kernel_label_list]
    long_median_properties = long_cluster_df.groupby(subcluster_level)[sliced_kernel_label_list].median()
    long_median_properties_list.append(long_median_properties)

In [ ]:
unhighlighted_subclusters = [item for item in long_median_properties.index.tolist() if item not in highlighted_subclusters]
# Generate a categorical colormap using colorcet
colors = cc.glasbey_light[:len(long_median_properties.index)]
# Create a dictionary mapping each category to a color
color_dict = dict(zip(long_median_properties.index, colors))
reorder_cluster_dict = {cluster:i for i,cluster in enumerate(long_median_properties.index.tolist())}
for subcluster in unhighlighted_subclusters:
    color_dict[subcluster] = '#808080'
categorical_colors = [color_dict[item] for item in long_median_properties.index.tolist()]
cmap = sns.color_palette(categorical_colors)
# normalize the data
norm = mpl.colors.Normalize(vmin=0, vmax=len(reorder_cluster_dict)-1)

In [ ]:
for time_slice_idx,time_slice in enumerate(time_slices):

    fig, axes = plt.subplots(1, len(labels)+1, figsize=(4, 4), sharey=True)
    fig.suptitle(time_slice_labels[time_slice_idx], fontsize=16)
    
    label_names = [r'$ \tau_{C} $',r'$ L_{Sept} $',r'$ L_{C} $',r'$ W_{C} $',r'$ I_{rpsL} $',r'$ \lambda $',]

    long_median_properties = long_median_properties_list[time_slice_idx]
    
    cluster_colors = {int(long_median_properties.index.tolist()[i]):vega_20_scanpy[i] for i in range(len(long_median_properties))}
    # white_cmap = mpl.colors.ListedColormap(['white'])

    # Create separate single-column heatmaps for each column
    sns.heatmap(np.array(range(len(long_median_properties))).reshape(-1,1),cmap=cmap,norm=norm,\
                cbar=False, yticklabels=False, xticklabels=False, ax=axes[0],annot=False,\
                   linewidths=0,linecolor="black", annot_kws={"fontsize":14})

    cbar_list = []
    for i, label in enumerate(labels):
        working_heatmap = sns.heatmap(long_median_properties.values[:,i:i+1],vmin=min_vals[i],vmax=max_vals[i],\
                    cmap="coolwarm",center=median_vals_dict[label],\
                   cbar=False, yticklabels=False, ax=axes[i+1],annot=False,\
                   linewidths=0.5,linecolor="black")
        axes[i+1].set_xticklabels(labels=[label_names[i]],fontsize=16)
            
    # Remove margins between subplots
    plt.subplots_adjust(wspace=0)
        
    # plt.savefig("/home/de64/group/CRISPRi_Libraries/dev_notebooks/2023-04-05_Multi_Dataset_Figures/Filament_Cluster_TimeSlice_" + str(time_slice_idx) + ".png",dpi=300,transparent=True)

In [ ]:
print(1)

### Luis: Matrix over time

In [ ]:
min_representatives = 10
cluster_level = "L1"
subcluster_level = "L2"
selected_clusters = ["4","5","18"]
highlighted_subclusters = ["12","17","25","30"]


In [ ]:
clust_lev = 'L2'
clust_num = '12'
a = short_expt_clustree_pandas_df.loc[short_expt_clustree_pandas_df[clust_lev] == clust_num, 'Kernel Trace: major_axis_length']
list_median = np.zeros(20)
for i in range(len(list_median)):
    list_median[i] = a.str[i].median()

In [ ]:
l12 = list_median

In [ ]:
clust_lev = 'L2'
clust_num = '17'
a = short_expt_clustree_pandas_df.loc[short_expt_clustree_pandas_df[clust_lev] == clust_num, 'Kernel Trace: major_axis_length']
list_median = np.zeros(20)
for i in range(len(list_median)):
    list_median[i] = a.str[i].median()
l17 = list_median

In [ ]:
clust_lev = 'L2'
clust_num = '25'
a = short_expt_clustree_pandas_df.loc[short_expt_clustree_pandas_df[clust_lev] == clust_num, 'Kernel Trace: major_axis_length']
list_median = np.zeros(20)
for i in range(len(list_median)):
    list_median[i] = a.str[i].median()
l25 = list_median

In [ ]:
clust_lev = 'L2'
clust_num = '30'
a = short_expt_clustree_pandas_df.loc[short_expt_clustree_pandas_df[clust_lev] == clust_num, 'Kernel Trace: major_axis_length']
list_median = np.zeros(20)
for i in range(len(list_median)):
    list_median[i] = a.str[i].median()
l30 = list_median

In [ ]:
clust_lev = 'L2'
clust_num = '3'
a = short_expt_clustree_pandas_df.loc[short_expt_clustree_pandas_df[clust_lev] == clust_num, 'Kernel Trace: major_axis_length']
list_median = np.zeros(20)
for i in range(len(list_median)):
    list_median[i] = a.str[i].median()
l3 = list_median

In [ ]:
clust_lev = 'L2'
clust_num = '1'
a = short_expt_clustree_pandas_df.loc[short_expt_clustree_pandas_df[clust_lev] == clust_num, 'Kernel Trace: major_axis_length']
list_median = np.zeros(20)
for i in range(len(list_median)):
    list_median[i] = a.str[i].median()
l1 = list_median

In [ ]:
plt.plot(l12)
plt.plot(l17)
plt.plot(l25)
plt.plot(l30)
plt.plot(l3)
plt.plot(l1)

In [ ]:
short_expt_clustree_pandas_df['Kernel Trace: major_axis_length']

In [ ]:
# Show the final timepoint with colors for the 

#### Testing Hierarchical Clustering (with genes)

In [ ]:
n_gene_counts_min = 3
percent_for_assignment = 0.3

hierarchical_df = long_cluster_df[long_cluster_df[subcluster_level].isin(["12","17","25","30"])]
gene_counts = hierarchical_df.groupby("Gene").size()
hierarchical_genes = gene_counts[gene_counts>=n_gene_counts_min].index.tolist()
hierarchical_df = hierarchical_df[hierarchical_df["Gene"].isin(hierarchical_genes)]

In [ ]:
sliced_kernel_label_list = ["8 Hours: " + kernel_label for kernel_label in kernel_label_list]
sliced_kernel_z_label_list = ["8 Hours: " + kernel_label for kernel_label in zscore_kernel_label_list]
gene_mean_df = hierarchical_df.groupby("Gene")[sliced_kernel_label_list + sliced_kernel_z_label_list].mean()

gene_cluster_assign_series = hierarchical_df.groupby("Gene")["L2"].apply(lambda x: \
                            x.value_counts()[x.value_counts()/len(x)>percent_for_assignment].index.tolist())
gene_mean_df["Cluster Assignment"] = gene_cluster_assign_series

## temp overwriting with a single assignment for convenience
gene_mean_df["Cluster Assignment"] = gene_mean_df["Cluster Assignment"].apply(lambda x: x[0])

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

In [ ]:
distance_matrix = pdist(gene_mean_df[sliced_kernel_z_label_list], metric='euclidean')
Z = linkage(distance_matrix, method='ward')

In [ ]:
# Create a custom layout for the plots
fig = plt.figure(figsize=(4, 10))
gs = gridspec.GridSpec(1, len(labels)+2, width_ratios=[1, 1, 1, 1, 1, 1, 1, 1])

# Plot the dendrogram
ax = plt.subplot(gs[0])
ax.axis('off')
dendrogram(Z, leaf_rotation=90., leaf_font_size=8., orientation='left', ax=ax, color_threshold=0,link_color_func=lambda x: 'black')

# Get the order of leaves in the dendrogram
leaves_order = dendrogram(Z, no_plot=True)['leaves']
reordered_df = gene_mean_df.iloc[leaves_order]

# fig, axes = plt.subplots(1, len(labels), figsize=(4, 8), sharey=True)

label_names = [r'$ \tau_{C} $',r'$ L_{Sept} $',r'$ L_{C} $',r'$ W_{C} $',r'$ I_{rpsL} $',r'$ \lambda $',]

cbar_list = []
for i, label in enumerate(labels):
    ax = plt.subplot(gs[i+1])
    
    # if i == len(labels)-1:    
    #     working_heatmap = sns.heatmap(reordered_df[sliced_kernel_label_list[i]].values.reshape(-1,1),vmin=min_vals[i],vmax=max_vals[i],\
    #                 cmap="coolwarm",center=median_vals_dict[label],\
    #                cbar=False, yticklabels=reordered_df.index, ax=ax,annot=False,\
    #                linewidths=0.5,linecolor="black")
    #     ax.tick_params(right=True, labelright=True, labelleft=False, labelrotation=0)
    
    working_heatmap = sns.heatmap(reordered_df[sliced_kernel_label_list[i]].values.reshape(-1,1),vmin=min_vals[i],vmax=max_vals[i],\
                cmap="coolwarm",center=median_vals_dict[label],\
               cbar=False, yticklabels=False, ax=ax,annot=False,\
               linewidths=0.5,linecolor="black")
    ax.set_xticklabels(labels=[label_names[i]],fontsize=16)
    
ax = plt.subplot(gs[len(labels)+1])
# Create separate single-column heatmaps for each column
sns.heatmap(reordered_df["Cluster Assignment"].apply(lambda x: reorder_cluster_dict[x]).values.reshape(-1,1).astype(int),\
            cmap=cmap,norm=norm,cbar=False, yticklabels=reordered_df.index, xticklabels=False, ax=ax,annot=False,\
               linewidths=0,linecolor="black", annot_kws={"fontsize":14})
ax.tick_params(right=True, labelright=True, labelleft=False, labelrotation=0)

# Remove margins between subplots
plt.subplots_adjust(wspace=0)
# plt.savefig("/home/de64/group/CRISPRi_Libraries/dev_notebooks/2023-04-05_Multi_Dataset_Figures/Filament_Cluster_Time_Dendrogram.png",dpi=300,\
            # transparent=True,bbox_inches = "tight")

In [ ]:
sns.heatmap(reordered_df[sliced_kernel_label_list], cmap='coolwarm', cbar=True)
plt.title('Heatmap of Reordered Vectors')
plt.xlabel('Dimensions')
plt.ylabel('Data Points')
plt.show()

In [ ]:
long_plot_df = short_expt_clustree_pandas_df.copy()
long_plot_df["In Long Cluster"] = False
long_plot_df.loc[long_cluster_df.index,"In Long Cluster"] = True
long_plot_df.loc[long_plot_df[~long_plot_df["In Long Cluster"]].index,subcluster_level] = np.NaN
long_plot_df[subcluster_level] =  long_plot_df[subcluster_level].cat.add_categories(["Other"])
long_plot_df.loc[long_plot_df[long_plot_df[subcluster_level].isnull()].index,subcluster_level] = "Other"
long_plot_df.loc[:,subcluster_level] = long_plot_df[subcluster_level].cat.remove_unused_categories().copy()

unique_categories = long_plot_df[subcluster_level].unique()
num_categories = len(unique_categories) # Subtract 1 to account for the 'missing' category
# Create a color palette with 'num_categories' colors
colors = sns.color_palette('tab10', n_colors=num_categories+1)
colors.remove((0.4980392156862745, 0.4980392156862745, 0.4980392156862745))
# Create a dictionary mapping categories to colors, while setting the grey color for the "missing" category
palette = {category: color for category, color in zip(unique_categories, colors)}
palette['Other'] = 'grey'

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(8, 9),sharex=False,sharey=False)
for i,time_slice_label in enumerate(time_slice_labels):
    
    sns.scatterplot(data=long_plot_df[~long_plot_df["In Long Cluster"]],x=time_slice_label + ': Kernel Trace: Instantaneous Growth Rate: Volume',\
                    y=time_slice_label + ': Kernel Trace: major_axis_length',s=10,hue="L2",palette=palette,ax=axes[i,0],legend=False)

    sns.scatterplot(data=long_plot_df[long_plot_df["In Long Cluster"]],x=time_slice_label + ': Kernel Trace: Instantaneous Growth Rate: Volume',\
                    y=time_slice_label + ': Kernel Trace: major_axis_length',s=10,hue="L2",palette=palette,ax=axes[i,0],legend=False)
    
    axes[i,0].set_ylabel("Length (um)")
    axes[i,0].set_xlabel("Growth Rate")
    
    axes[i,0].set_ylim(2,14)
    axes[i,0].set_xlim(0.,1.3)
    
for i,time_slice_label in enumerate(time_slice_labels):
    
    if i==0:
        sns.scatterplot(data=long_plot_df[~long_plot_df["In Long Cluster"]],x=time_slice_label + ': Kernel Trace: Septum Displacement Length Normalized',\
                        y=time_slice_label + ': Kernel Trace: major_axis_length',s=10,hue="L2",palette=palette,ax=axes[i,1],legend=True)
        
    else:
        sns.scatterplot(data=long_plot_df[~long_plot_df["In Long Cluster"]],x=time_slice_label + ': Kernel Trace: Septum Displacement Length Normalized',\
                y=time_slice_label + ': Kernel Trace: major_axis_length',s=10,hue="L2",palette=palette,ax=axes[i,1],legend=False)

    
    sns.scatterplot(data=long_plot_df[long_plot_df["In Long Cluster"]],x=time_slice_label + ': Kernel Trace: Septum Displacement Length Normalized',\
                    y=time_slice_label + ': Kernel Trace: major_axis_length',s=10,hue="L2",palette=palette,ax=axes[i,1],legend=False)
    
    axes[i,1].set_ylabel("Length (um)")
    axes[i,1].set_xlabel("Septum Displacement")
    
    axes[i,1].set_ylim(2,14)
    axes[i,1].set_xlim(0.,0.1)
    
plt.show()

# sns.scatterplot(data=long_plot_df,x='Mean: Septum Displacement Length Normalized',y='Mean: major_axis_length',\
#                 s=10,hue="L2")
# plt.show()

In [ ]:
par_like = long_cluster_df[long_cluster_df["L2"].isin(['23'])].copy()
par_plot_df = short_expt_clustree_pandas_df.copy()
par_plot_df["Paritioning"] = False
par_plot_df.loc[par_like.index,"Paritioning"] = True

In [ ]:
sns.scatterplot(data=par_plot_df,x='Mean: Instantaneous Growth Rate: Volume',y='Mean: major_axis_length',\
                s=10,hue="Paritioning")
plt.show()
sns.scatterplot(data=par_plot_df,x='Mean: Septum Displacement Length Normalized',y='Mean: major_axis_length',\
                s=10,hue="Paritioning")
plt.show()

#### Note for next steps

- Outline
    - A - General clustering of the whole dataset with mean phenotypes
    - B - Length defects
        - Strong effects can be pulled out and divisome recovered (label in scatter)
        - Partitioning functions can be distinguished by the error in division site placement
    - C - Width defects
        - Strong effects, as before
        - Lol system distinguished by division site placement AND kinetics
    - D?- Other Major CLusters
        - thyA cluster
    - E - Ribosome and tRNA clusters
        - Reveals trend exploired in next figure

- Representations
    - Heatmap of median cluster values at t=6, 8, 10 hrs
    - Scatter plot of binary relations that seperate data (10 hrs)
    

- Try using the final timepoint instead of "steady state" or even the max disturbence
    - I should use a timeseries representation for this, it seems necssary in some cases
    - Nice to show the lol system is distinguished
    - Scatters can be 
- Decide what I want to do with this figure...I could tell a story about gene association...
    - Beyond the large clusting, what is necessary here...seems that a table would do
    - The lower resolution clustering analysis should be specifically to make claims about division and partitioning (which are sort of interesting)
    - Lower cluterings also provide "stronger" groups
    - Perhaps that with a nice overlay of the group of interest...maybe a cartoon of the phenotype
- Mock up the main subclusters

In [ ]:
wide_cluster_df = clusters_df[clusters_df["L0"]=="6"].copy()
wide_cluster_df.loc[:,"L2"] = wide_cluster_df["L2"].cat.remove_unused_categories().copy()
wide_median_properties = wide_cluster_df.groupby("L2")[labels].median()

In [ ]:
fig, axes = plt.subplots(1, len(labels)+1, figsize=(4, 4), sharey=True)

label_names = [r'$ \tau_{C} $',r'$ L_{Sept} $',r'$ L_{C} $',r'$ W_{C} $',r'$ I_{rpsL} $',r'$ \lambda $',]

cluster_colors = {int(wide_median_properties.index.tolist()[i]):vega_20_scanpy[i] for i in range(len(wide_median_properties))}
white_cmap = mpl.colors.ListedColormap(['white'])

# Create separate single-column heatmaps for each column
sns.heatmap(np.array(wide_median_properties.index.tolist()).reshape(-1,1).astype(int),cmap=white_cmap,\
            cbar=False, yticklabels=False, xticklabels=False, ax=axes[0],annot=True,\
               linewidths=0,linecolor="black", annot_kws={"fontsize":14})

cbar_list = []
for i, label in enumerate(labels):
    working_heatmap = sns.heatmap(wide_median_properties.values[:,i:i+1],vmin=min_vals[i],vmax=max_vals[i],\
                cmap="coolwarm",center=median_vals_dict[label],\
               cbar=False, yticklabels=False, ax=axes[i+1],annot=False,\
               linewidths=0.5,linecolor="black")
    axes[i+1].set_xticklabels(labels=[label_names[i]],fontsize=16)
# Remove margins between subplots
plt.subplots_adjust(wspace=0)

In [ ]:
plt.plot(np.array(wide_cluster_df.groupby("L2")['Kernel Trace: Septum Displacement Length Normalized'].mean().tolist()).T)
plt.show()
plt.plot(np.array(wide_cluster_df.groupby("L2")['Kernel Trace: minor_axis_length'].mean().tolist()).T)
plt.show()
plt.plot(np.array(wide_cluster_df.groupby("L2")['Kernel Trace: Instantaneous Growth Rate: Volume'].mean().tolist()).T)
plt.show()
plt.plot(np.array(wide_cluster_df.groupby("L2")['Kernel Trace: major_axis_length'].mean().tolist()).T)
plt.show()

In [ ]:
plt.plot(np.array(wide_cluster_df.groupby("L2")['Kernel Trace: minor_axis_length'].mean().tolist()).T)

In [ ]:
elongasome_like = wide_cluster_df[wide_cluster_df["L2"].isin(['2','5','12'])].copy()

In [ ]:
elongasome_like["Gene"].unique()

In [ ]:
interesting_df = wide_cluster_df[wide_cluster_df["L2"].isin(['25'])].copy()

In [ ]:
seq = interesting_df[interesting_df["Gene"]=="gltB"]["sgRNA"].apply(lambda x: x[9:]).iloc[0]
interesting_df[interesting_df["sgRNA"].apply(lambda x: x[9:])==seq]

In [ ]:
interesting_df[interesting_df["Gene"]=="yihW"]

In [ ]:
interesting_df[interesting_df["Gene"]=="gltB"]["sgRNA"].apply(lambda x: x[:10]).iloc[0]

In [ ]:
np.unique(interesting_df["Gene"],return_counts=True)

In [ ]:
plt.plot(wide_cluster_df.groupby("L2")['Kernel Trace: Septum Displacement Length Normalized'].mean()[5])
plt.plot(wide_cluster_df.groupby("L2")['Kernel Trace: Septum Displacement Length Normalized'].mean()[7])

In [ ]:
wide_cluster_df.groupby("L2")['Kernel Trace: Septum Displacement Length Normalized'].mean()

In [ ]:
sns.scatterplot(data=wide_cluster_df,x='Mean: Septum Displacement Length Normalized',y='Mean: minor_axis_length',\
                s=10,hue="L2")
plt.show()
sns.scatterplot(data=wide_cluster_df,x='Mean: major_axis_length',y='Mean: minor_axis_length',\
                s=10,hue="L2")
plt.show()
sns.scatterplot(data=wide_cluster_df,x='Mean: Instantaneous Growth Rate: Volume',y='Mean: minor_axis_length',\
                s=10,hue="L2")
plt.show()